# Análise do Winning the Race — America's AI Action Plan

Este notebook integra uma análise acadêmica de rigor, desenvolvida no âmbito de uma pesquisa em Relações Internacionais dedicada ao estudo comparado das estratégias nacionais de inteligência artificial. Seu objeto específico é o documento "Winning the Race: America's AI Action Plan", publicado pela Casa Branca, que estabelece a estratégia dos Estados Unidos para a liderança global em inteligência artificial.

A análise aqui conduzida busca compreender, de modo simultaneamente **quantitativo e qualitativo**, a linguagem empregada pelo documento — os termos, categorias, ênfases retóricas, valores e prioridades estratégicas que estruturam o texto —, de modo a identificar os marcos conceituais e os campos semânticos por meio dos quais os Estados Unidos formula sua política de inteligência artificial.

A partir dessa leitura, pretende-se **posicionar o documento internacionalmente**, comparando a linguagem e as escolhas discursivas de os Estados Unidos com o vocabulário e as ênfases adotados pelos demais países e blocos contemplados neste projeto (Brasil, China, Estados Unidos, Europa e Índia), de modo a mapear convergências, divergências e posicionamentos estratégicos distintivos no debate internacional sobre desenvolvimento e governança de inteligência artificial.

As análises e visualizações produzidas neste notebook seguem as diretrizes metodológicas da **Skill02** (Análise e Visualização Gráfica de Documentos): baseiam-se exclusivamente no campo `texto_completo` do JSON de extração correspondente a este documento, produzido na etapa anterior (Skill01), adotam rigor acadêmico integral na leitura do texto-fonte, evitam generalizações, simplificações e inferências não fundamentadas no texto original, e cada visualização construída é acompanhada de sua respectiva descrição, leitura, interpretação e eventuais limitações metodológicas.


## Análise de Vocabulário — Termos mais Frequentes

Esta seção aplica a **Skill02** (Análise e Visualização Gráfica de Documentos) em conjunto com seu complemento especializado, a **Skill 02_Análise_Vocab_A** (Análise de Vocabulário e Termos), ao JSON `americas_ai_action_plan.json`, produzido na etapa anterior (Skill01).

**Protocolo de uso do JSON.** Conforme exigido pela Skill02, apenas os campos `titulo`, `pais_ou_bloco` e `texto_completo` são utilizados nesta análise. Os campos `elementos_descartados`, `data_extracao`, `data_publicacao` e `fonte` não entram, em nenhuma hipótese, na contagem ou na interpretação do vocabulário.

**Idioma identificado.** O documento está integralmente redigido em **inglês**.

> **Revisão metodológica (2026-08-05).** Por determinação do usuário, esta seção foi revisada para corrigir a regra cega de tokenização que tratava todo hífen como separador, e para desambiguar contextualmente termos polissêmicos — a própria versão anterior já apontava, como limitação, que "generation" misturava "geração de energia" (*power generation*) e "nova geração tecnológica" (*next-generation*) sob um único token, exatamente o tipo de erro que o usuário pediu para corrigir em todo o projeto. Foi também expandido o corte para **Top 50**. O registro persistente documenta a versão anterior e a mudança em detalhe.

**Etapas metodológicas aplicadas** (ver célula de código a seguir para a implementação exata):

1. **Extração bruta** de todos os termos de `texto_completo`, sem qualquer corte prévio de conteúdo.
2. **Pré-processamento do texto corrido, antes da tokenização:** `"U.S."` → `"United States"` (bigrama protegido, 50 ocorrências); `"Artificial Intelligence"` → `"AI"`; remoção do subtítulo estrutural fixo `"Recommended Policy Actions"` (30 ocorrências, rótulo editorial repetitivo). Três desambiguações contextuais de "power"/"generation" protegidas por locução: `"power generation"`/`"energy generation"` (sentido de infraestrutura elétrica), `"computing power"` (sentido computacional) e `"balance of power"`/`"power of American innovation"` (sentido geopolítico/abstrato) — três referentes distintos que a versão anterior fundia sob os tokens genéricos "power" e "generation".
3. **Tokenização com hífen preservado no token** (`[A-Za-z]+(?:-[A-Za-z]+)*`) — não mais "hífen = separador universal". Das **83 formas hifenizadas** identificadas em `texto_completo`, todas foram mantidas unidas: cada uma nomeia um conceito técnico, categoria de política, programa ou nome próprio mais específico do que a soma das partes — auditoria completa, organizada tematicamente (descritores "AI-X", ecossistema aberto, categorias "high-X", programas de força de trabalho, nomes de leis/instituições etc.), no registro persistente.
4. **Desambiguação contextual de termos polissêmicos/homônimos** (nova etapa obrigatória): além de "power"/"generation" (passo 2), "state(s)" (subnacional) segue verificado como referente distinto de "United States".
5. **Remoção (exclusão)** de palavras funcionais do inglês, tokens de um único caractere e do subtítulo estrutural — lista integral no registro persistente.
6. **Normalização/agrupamento** de variantes que designam exatamente o mesmo referente: 69 pares singular/plural de substantivos (35 já existentes + 34 novos nesta revisão) e 30 famílias verbais (12 já existentes + 18 novas). Pares mantidos deliberadamente **não agrupados** (herdados, reconfirmados): *America*/*American*/*Americans*; *research*/*researchers*; *development*/*develop*; *security*/*secure*; *state(s)*/*United States*; *provide*/*providers*; *economic*/*economy*; *advance*/*advanced*.
7. **Critério de corte:** **Top 50** termos por frequência absoluta (critério padrão da Skill 02_Análise_Vocab_A), com o **Top 25** também apresentado como recorte adicional.

O registro completo, persistente e auditável — incluindo a auditoria das 83 formas hifenizadas, a tabela de desambiguação contextual e a lista integral de agrupamentos — está em [`registro_vocabulario_americas_ai_action_plan.md`](./registro_vocabulario_americas_ai_action_plan.md), nesta mesma pasta.


In [ ]:
import json, re
from collections import Counter
import pandas as pd

with open("americas_ai_action_plan.json", encoding="utf-8") as f:
    doc = json.load(f)

titulo, pais, texto = doc["titulo"], doc["pais_ou_bloco"], doc["texto_completo"]
print(f"Documento: {titulo} | País/bloco: {pais} | Caracteres em texto_completo: {len(texto)}")

STOPWORDS_EN = set("""
a an the
and or but nor so yet if because while although that which who whom whose when where how than whether as
this these those it its they their them theirs he she his her him himself herself itself themselves
we our ours us you your yours i my mine
is are was were be been being am
has have had having
do does did doing
will would shall should can could may might must
to of in on at by for with from into through across throughout under over without within among between via per about upon toward towards
not no nor
also more most many much very only further therefore thus however moreover
there here
such other others any all both each every some
including include includes included
etc eg ie
""".split())

STRUCTURAL_PHRASES_REMOVIDAS = [r'\bRecommended Policy Actions\b']

def preprocess(text):
    text = text.replace("U.S.", "United States")
    text = re.sub(r'\bArtificial Intelligence\b', 'AI', text, flags=re.I)
    for pat in STRUCTURAL_PHRASES_REMOVIDAS:
        text = re.sub(pat, ' ', text)
    text = re.sub(r'\bUnited States\b', ' UNITEDSTATESTOKEN ', text)
    # desambiguação contextual (registro persistente, Seção 4): "power"/"generation"
    # misturavam sentido energético, computacional e geopolítico sob um único token
    text = re.sub(r'\bpower\s+generation\b', ' POWERGENTOKEN ', text, flags=re.I)
    text = re.sub(r'\benergy\s+generation\b', ' POWERGENTOKEN ', text, flags=re.I)
    text = re.sub(r'\bcomputing\s+power\b', ' POWERCOMPUTETOKEN ', text, flags=re.I)
    text = re.sub(r'\bbalance\s+of\s+power\b', ' POWERGEOTOKEN ', text, flags=re.I)
    text = re.sub(r'\bpower\s+of\s+American\s+innovation\b', ' POWERGEOTOKEN ', text, flags=re.I)
    return text

texto_tratado = preprocess(texto)

# Tokenização (Skill 02_Análise_Vocab_A, item 4): hífen preservado como parte do token —
# cada composto hifenizado é candidato a token único, sujeito à auditoria individual
# registrada no registro persistente (Seção 2). Das 83 formas identificadas neste
# documento, todas foram mantidas unidas (nenhuma regra cega em nenhuma direção).
tokens_brutos = re.findall(r"[A-Za-z]+(?:-[A-Za-z]+)*", texto_tratado)
tokens_lower = [t.lower() for t in tokens_brutos]
tokens_filtrados = [t for t in tokens_lower if len(t) > 1 and t not in STOPWORDS_EN]

NOUN_MERGES = {
    "systems": ["system", "systems"], "models": ["model", "models"], "agencies": ["agency", "agencies"],
    "technology": ["technology", "technologies"], "programs": ["program", "programs"], "workers": ["worker", "workers"],
    "controls": ["control", "controls"], "standards": ["standard", "standards"], "capabilities": ["capability", "capabilities"],
    "tools": ["tool", "tools"], "actions": ["action", "actions"], "risks": ["risk", "risks"],
    "developers": ["developer", "developers"], "initiatives": ["initiative", "initiatives"], "stakeholders": ["stakeholder", "stakeholders"],
    "frameworks": ["framework", "frameworks"], "resources": ["resource", "resources"], "occupations": ["occupation", "occupations"],
    "assessments": ["assessment", "assessments"], "evaluations": ["evaluation", "evaluations"], "regulations": ["regulation", "regulations"],
    "values": ["value", "values"], "threats": ["threat", "threats"], "vulnerabilities": ["vulnerability", "vulnerabilities"],
    "centers": ["center", "centers"], "innovation": ["innovation", "innovations"], "needs": ["need", "needs"],
    "skills": ["skill", "skills"], "countries": ["country", "countries"], "efforts": ["effort", "efforts"],
    "employers": ["employer", "employers"], "industry": ["industry", "industries"], "partners": ["partner", "partners"],
    "sector": ["sector", "sectors"], "services": ["service", "services"], "sources": ["source", "sources"],
    "states (subnacionais, distinto de United States)": ["state", "states"],
    # -- novos pares (cobertura ampliada para Top 50) --
    "fields": ["field", "fields"], "benefits": ["benefit", "benefits"], "pillars": ["pillar", "pillars"],
    "semiconductors": ["semiconductor", "semiconductors"], "applications": ["application", "applications"],
    "jobs": ["job", "jobs"], "goals": ["goal", "goals"], "laws": ["law", "laws"],
    "businesses": ["business", "businesses"], "rules": ["rule", "rules"], "orders": ["order", "orders"],
    "governments": ["government", "governments"], "academics": ["academic", "academics"],
    "markets": ["market", "markets"], "investments": ["investment", "investments"],
    "projects": ["project", "projects"], "approaches": ["approach", "approaches"], "chips": ["chip", "chips"],
    "environments": ["environment", "environments"], "processes": ["process", "processes"],
    "workflows": ["workflow", "workflows"], "deepfakes": ["deepfake", "deepfakes"], "ways": ["way", "ways"],
    "officers": ["officer", "officers"], "nations": ["nation", "nations"], "agendas": ["agenda", "agendas"],
    "allies": ["ally", "allies", "allied"], "challenges": ["challenge", "challenges"], "councils": ["council", "councils"],
    "protections": ["protection", "protections"], "requirements": ["requirement", "requirements"],
    "reviews": ["review", "reviews"], "roles": ["role", "roles"], "providers": ["provider", "providers"],
}

VERB_MERGES = {
    "lead / led": ["lead", "leads", "leading", "led"],
    "develop": ["develop", "develops", "developing", "developed"],
    "create": ["create", "creates", "creating", "created"],
    "build": ["build", "builds", "building", "built"],
    "establish": ["establish", "establishes", "establishing", "established"],
    "ensure": ["ensure", "ensures", "ensuring", "ensured"],
    "promote": ["promote", "promotes", "promoting", "promoted"],
    "expand": ["expand", "expands", "expanding", "expanded"],
    "support": ["support", "supports", "supporting", "supported"],
    "make": ["make", "makes", "making", "made"],
    "require": ["require", "requires", "requiring", "required"],
    "use": ["use", "uses", "used", "using"],
    "adopt": ["adopt", "adopts", "adopting", "adopted"],
    "align": ["align", "aligns", "aligning", "aligned"],
    "conduct": ["conduct", "conducts", "conducting", "conducted"],
    "deliver": ["deliver", "delivers", "delivering", "delivered"],
    "design": ["design", "designs", "designing", "designed"],
    "direct": ["direct", "directs", "directing", "directed"],
    "fund": ["fund", "funds", "funding", "funded"],
    "implement": ["implement", "implements", "implementing", "implemented"],
    "increase": ["increase", "increases", "increasing", "increased"],
    "launch": ["launch", "launches", "launching", "launched"],
    "maintain": ["maintain", "maintains", "maintaining", "maintained"],
    "measure": ["measure", "measures", "measuring", "measured"],
    "pilot": ["pilot", "pilots", "piloting", "piloted"],
    "protect": ["protect", "protects", "protecting", "protected"],
    "streamline": ["streamline", "streamlines", "streamlining", "streamlined"],
    "supply": ["supply", "supplies", "supplying", "supplied"],
    "train": ["train", "trains", "training", "trained"],
    "transform": ["transform", "transforms", "transforming", "transformed"],
}

canon = {}
for label, variants in NOUN_MERGES.items():
    for v in variants:
        canon[v] = label
for label, variants in VERB_MERGES.items():
    for v in variants:
        canon[v] = label
canon["ai"] = "AI"
canon["unitedstatestoken"] = "United States"
canon["doc"] = "DOC"
canon["powergentoken"] = "power/energy generation (infraestrutura elétrica)"
canon["powercomputetoken"] = "power (computacional)"
canon["powergeotoken"] = "power (geopolítico/abstrato)"
canon["power"] = "power (energia/infraestrutura elétrica)"
canon["generation"] = "generation (tecnológica)"

tokens_finais = [canon.get(t, t) for t in tokens_filtrados]
freq = Counter(tokens_finais)

print(f"Tokens brutos: {len(tokens_brutos)} | após remoção de funcionais/resíduos: {len(tokens_filtrados)} | termos distintos no vocabulário final: {len(freq)}")

DISPLAY_CAPITALIZE = {"american": "American", "america": "America", "trump": "Trump"}

CORTE = 50
top_termos_raw = freq.most_common(CORTE)
top_termos = [(DISPLAY_CAPITALIZE.get(t, t), n) for t, n in top_termos_raw]
top_25 = top_termos[:25]

tabela_top50 = pd.DataFrame(top_termos, columns=["termo", "frequência"])
tabela_top50.index = tabela_top50.index + 1
tabela_top50


In [ ]:
import matplotlib.pyplot as plt

termos_plot = [t for t, _ in top_25][::-1]
freqs_plot = [n for _, n in top_25][::-1]

fig, ax = plt.subplots(figsize=(9, 10))
barras = ax.barh(termos_plot, freqs_plot, color="#1f4e79", height=0.68)

ax.set_xlabel("Frequência absoluta (nº de ocorrências em texto_completo)", fontsize=11)
ax.set_title(
    "Termos mais frequentes — America's AI Action Plan (EUA, 2025)\n"
    "Top 25 termos, após correção de hífen/bigramas/polissemia (Skill 02_Análise_Vocab_A)",
    fontsize=12.5, fontweight="bold", loc="left", pad=14
)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="y", labelsize=10.5)
ax.tick_params(axis="x", labelsize=9.5)
ax.set_axisbelow(True)
ax.xaxis.grid(True, color="#d9d9d9", linewidth=0.8)

for bar, val in zip(barras, freqs_plot):
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height() / 2, str(val),
            va="center", ha="left", fontsize=9.5, color="#333333")

ax.set_xlim(0, max(freqs_plot) * 1.12)
fig.text(0.02, -0.01,
         "Fonte: texto_completo de americas_ai_action_plan.json (Skill01), tratado segundo a Skill02 / Skill 02_Análise_Vocab_A.",
         fontsize=8, color="#666666")
fig.tight_layout()
plt.show()


### Visão expandida — Top 50

Por determinação do usuário, a análise passa a cobrir também o **Top 50** (o Top 25 acima é mantido por continuidade). A expansão é sustentada pela cobertura ampliada de normalização morfológica — 69 grupos de substantivos (35 já existentes + 34 novos) e 30 famílias verbais (12 + 18 novas) — descrita no registro persistente, Seção 3.


In [ ]:
termos_plot50 = [t for t, _ in top_termos][::-1]
freqs_plot50 = [n for _, n in top_termos][::-1]
cores50 = ["#1f4e79" if i < 25 else "#8fb3d1" for i in range(len(termos_plot50))][::-1]

fig, ax = plt.subplots(figsize=(9.5, 18))
barras = ax.barh(termos_plot50, freqs_plot50, color=cores50, height=0.72)

ax.set_xlabel("Frequência absoluta (nº de ocorrências em texto_completo)", fontsize=11)
ax.set_title(
    "Termos mais frequentes — America's AI Action Plan (EUA, 2025)\n"
    "Top 50 termos (visão expandida) — Skill02 + Skill 02_Análise_Vocab_A",
    fontsize=12.5, fontweight="bold", loc="left", pad=14
)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="y", labelsize=9.5)
ax.tick_params(axis="x", labelsize=9.5)
ax.set_axisbelow(True)
ax.xaxis.grid(True, color="#d9d9d9", linewidth=0.8)
ax.axhline(24.5, color="#999999", linestyle="--", linewidth=0.9)
ax.text(max(freqs_plot50) * 0.98, 25.3, "Top 25", ha="right", fontsize=8.5, color="#666666", style="italic")

for bar, val in zip(barras, freqs_plot50):
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height() / 2, str(val),
            va="center", ha="left", fontsize=8.5, color="#333333")

ax.set_xlim(0, max(freqs_plot50) * 1.12)
fig.text(0.02, -0.005,
         "Fonte: texto_completo de americas_ai_action_plan.json (Skill01), tratado segundo a Skill02 / Skill 02_Análise_Vocab_A. "
         "Corte de Top 50 (Skill 02_Análise_Vocab_A, item 6).",
         fontsize=8, color="#666666")
fig.tight_layout()
plt.show()


## Leitura, Interpretação e Limitações dos Gráficos

**O que foi construído.** Dois gráficos de barras horizontais: Top 25 (mantido por continuidade) e Top 50 (visão expandida, posições 26–50 em tom mais claro), após a correção metodológica descrita acima.

**O que mudou com a correção (comparação com a versão anterior).**
- **"AI" caiu de 269 para 251 ocorrências** — a diferença vinha do "AI" solto de compostos como "AI-related"(5), "AI-enabled"(4), "AI-driven"(2), "AI-specific"(2), "AI-ready"(1), "AI-based"(1), "AI-generated"(1), agora corretamente absorvidos em seus próprios tokens.
- **"open" (que reunia fragmentos de "open-source" e "open-weight") desaparece do vocabulário como token isolado** — "open-source" (4) e "open-weight" (4) agora são tokens próprios e auditáveis, exatamente a limitação já apontada na versão anterior deste registro.
- **"power" e "generation" (citados na versão anterior como o principal exemplo de "polissemia não desambiguada" deste documento) foram desdobrados em quatro rótulos**: `power (energia/infraestrutura elétrica)` (14 — "power grid", "power lines", "power markets"), `power/energy generation (infraestrutura elétrica)` (6 — "power generation", "energy generation"), `power (computacional)` (1 — "computing power") e `power (geopolítico/abstrato)` (2 — "balance of power", "power of American innovation") — quatro referentes que a contagem anterior misturava sob dois tokens genéricos.
- **A expansão para Top 50 revela um perfil vocabular mais completo:** "innovation" (21), "industry" (20), "DOD" (20), "workforce" (20), "grid" (18), "administration"/"manufacturing"/"tools"/"centers"/"CAISI"/"critical" (17), "semiconductors" (16), "NIST" (16), "science"/"fund"/"collaboration" (15).

**Interpretação à luz do documento.**
- **"AI" (251)** domina como esperado; valor discriminante baixo isoladamente.
- **"lead / led" (59)**, ainda majoritariamente formulaico ("Led by [órgão federal]..."), confirma a arquitetura interagencial do plano — mesma leitura da versão anterior, agora sobre uma contagem mais precisa.
- **"United States" (50) e "DOC" (50)**, empatados, com "federal" (49), "national" (44), "governments" (30), evidenciam a centralidade do aparato federal.
- **"grid" (18) e "power (energia/infraestrutura elétrica)" (14) somados a "power/energy generation" (6)** — visíveis apenas com a desambiguação aplicada — confirmam que a política energética é um eixo autônomo e substancial do plano (Pilar II, infraestrutura), e não um apêndice do vocabulário de "power" geopolítico, que agora aparece isolado e corretamente marginal (2 ocorrências).
- **"open-source" (4) e "open-weight" (4)**, agora tokens próprios (ainda que abaixo do corte de 50), tornam auditável o mesmo mecanismo de governança tecnológica identificado nos dois documentos chineses deste projeto (`ai_plus.json`, `new_generation_ai_development_plan.json`) — uma base concreta para comparação futura na pasta "Análise Conjunta".
- **"American" (35) e "America" (25)**, mantidos separados, somados ultrapassariam "federal" — ênfase retórica nacionalista/identitária.

**Ajustes possíveis.** (a) Gráfico separando siglas/órgãos (DOC, DOD, NIST, CAISI...) de termos temático-conceituais; (b) segmentação por Pilar (I/II/III); (c) frequência relativa ao lado da absoluta.

**Limitações remanescentes.**
- **Cobertura de normalização e desambiguação concentrada nos candidatos plausíveis ao Top 50** — não há varredura exaustiva de toda a cauda longa de 1.681 termos distintos.
- **"lead/led" como categoria mista** — agrega uso formulaico e usos esporádicos de sentido mais amplo de liderança; herdado da versão anterior, não afetado por esta revisão (não é um caso de referentes incompatíveis como "power"/"generation", mas de um mesmo campo semântico com pesos de uso desiguais).
- **"America"/"American"/"Americans" seguem não fundidos**, por representarem classes gramaticais e referentes distintos.

A metodologia completa está documentada em [`registro_vocabulario_americas_ai_action_plan.md`](./registro_vocabulario_americas_ai_action_plan.md).
